# 09 - Silver: Macro Annual Fact Table

This notebook creates `silver.fact_macro_annual`, one row per country-year with
GDP, trade openness, fiscal, and macro indicators sourced from IMF WEO.

**Input:**

- `bronze.imf_weo_raw`
- `silver.dim_country`
- `silver.dim_bloc_membership`

**Output:** `silver.fact_macro_annual`

**WEO indicator mapping:**

| WEO code | silver column |
|---|---|
| NGDPD | gdp_current_usd plus gdp_current_usd_billions display column |
| LP | population plus population_millions display column |
| NGDPDPC | gdp_per_capita_current_usd |
| NGDP_RPCH | real_gdp_growth_pct_imf |
| PCPIPCH | inflation_cpi_pct |
| GGXWDG_NGDP | gross_debt_pct_gdp_imf |
| GGR_NGDP | government_revenue_pct_gdp_imf |
| GGX_NGDP | government_expenditure_pct_gdp_imf |
| GGXCNL_NGDP | net_lending_borrowing_pct_gdp_imf |
| BCA_NGDPD | current_account_balance_pct_gdp_imf |

> **Unit contract:** `silver.fact_macro_annual.gdp_current_usd` must land at
> actual current USD scale, with `gdp_current_usd_billions` as the dashboard
> convenience display column. Cell 4 enforces that contract with an empirical
> Nigeria GDP scale anchor so a 1e9 scaling error fails loudly.


In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

loaded_at = datetime.now(timezone.utc)

# WEO indicator codes → silver column names (pivot keys)
INDICATOR_COLUMNS = {
    "NGDPD":       "gdp_current_usd_billions",
    "LP":          "population_millions",
    "NGDPDPC":     "gdp_per_capita_current_usd",
    "NGDP_RPCH":   "real_gdp_growth_pct_imf",
    "PCPIPCH":     "inflation_cpi_pct",
    "GGXWDG_NGDP": "gross_debt_pct_gdp_imf",
    "GGR_NGDP":    "government_revenue_pct_gdp_imf",
    "GGX_NGDP":    "government_expenditure_pct_gdp_imf",
    "GGXCNL_NGDP": "net_lending_borrowing_pct_gdp_imf",
    "BCA_NGDPD":   "current_account_balance_pct_gdp_imf",
}

print(f"Catalog: cemac_ecowas_aes_trade")
print(f"Output:  silver.fact_macro_annual")
print(f"Loaded at: {loaded_at.isoformat()}")
print(f"Indicator mapping: {len(INDICATOR_COLUMNS)} indicators")


In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0

required_tables = [
    ("bronze", "imf_weo_raw"),
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("WEO bronze input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.imf_weo_raw
""").show()

print("Per-indicator availability:")
spark.sql("""
    SELECT
        indicator_code,
        unit,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS min_year,
        MAX(year) AS max_year
    FROM bronze.imf_weo_raw
    GROUP BY indicator_code, unit
    ORDER BY indicator_code
""").show(truncate=False)

# Warn about population/per-capita if missing from bronze
available_codes = [
    r.indicator_code
    for r in spark.sql("SELECT DISTINCT indicator_code FROM bronze.imf_weo_raw").collect()
]
for missing_code in ["LP", "NGDPDPC"]:
    if missing_code not in available_codes:
        print(f"WARNING: {missing_code} not in bronze.imf_weo_raw — re-run imf_weo_extract.py and notebook 06 to populate.")


In [ ]:
# Cell 3 - Pivot WEO long → wide (one row per country-year)
weo_raw = (
    spark.table("bronze.imf_weo_raw")
    .where(F.col("indicator_code").isin(list(INDICATOR_COLUMNS.keys())))
    .select(
        F.upper(F.trim(F.col("country_iso3"))).alias("country_iso3"),
        F.col("year").cast("int").alias("year"),
        F.upper(F.trim(F.col("indicator_code"))).alias("indicator_code"),
        F.col("value").cast("double").alias("value"),
    )
)

pivot_exprs = [
    F.max(F.when(F.col("indicator_code") == indicator_code, F.col("value"))).alias(column_name)
    for indicator_code, column_name in INDICATOR_COLUMNS.items()
]

weo_wide_df = (
    weo_raw.groupBy("country_iso3", "year")
    .agg(*pivot_exprs)
)

print(f"Pivot complete: {weo_wide_df.count()} country-year rows")
weo_wide_df.orderBy("country_iso3", "year").show(5, truncate=False)


In [ ]:
# Cell 4 - Join dimensions and derive calculated columns
country_dim = spark.table("silver.dim_country").select(
    "country_key",
    "country_iso3",
    "country_name",
    F.col("region").alias("project_region"),
)

bloc_df = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("bloc_country_iso3"),
    F.col("bloc_code").alias("analytical_bloc_code"),
    F.col("bloc_name").alias("analytical_bloc_name"),
    F.col("valid_from").alias("bloc_valid_from"),
    F.col("valid_to").alias("bloc_valid_to"),
)

fact_df = (
    weo_wide_df.alias("w")
    .join(country_dim.alias("c"), F.col("w.country_iso3") == F.col("c.country_iso3"), "left")
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .join(
        bloc_df.alias("b"),
        (F.col("w.country_iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("year_end_date") >= F.col("b.bloc_valid_from"))
        & (F.col("b.bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("b.bloc_valid_to"))),
        "left",
    )
    # Bronze WEO values are normalized to actual USD and persons using the source scale.
    # Keep dashboard convenience columns in billions/millions, but do not rescale the base facts again.
    .withColumn("gdp_current_usd", F.col("w.gdp_current_usd_billions"))
    .withColumn("population", F.col("w.population_millions"))
    .withColumn("gdp_current_usd_billions_display", F.col("gdp_current_usd") / F.lit(1_000_000_000.0))
    .withColumn("population_millions_display", F.col("population") / F.lit(1_000_000.0))
    .withColumn(
        "gross_debt_usd",
        F.when(
            F.col("w.gross_debt_pct_gdp_imf").isNotNull() & F.col("gdp_current_usd").isNotNull(),
            F.col("w.gross_debt_pct_gdp_imf") / 100.0 * F.col("gdp_current_usd"),
        )
    )
    .withColumn("loaded_at", F.lit(str(loaded_at)))
    .select(
        F.col("c.country_key"),
        F.col("w.country_iso3"),
        F.col("c.country_name"),
        F.col("c.project_region"),
        F.col("b.analytical_bloc_code"),
        F.col("b.analytical_bloc_name"),
        F.col("w.year"),
        F.col("gdp_current_usd_billions_display").alias("gdp_current_usd_billions"),
        F.col("gdp_current_usd"),
        F.col("w.gdp_per_capita_current_usd"),
        F.col("population_millions_display").alias("population_millions"),
        F.col("population"),
        F.col("w.real_gdp_growth_pct_imf"),
        F.col("w.inflation_cpi_pct"),
        F.col("w.gross_debt_pct_gdp_imf"),
        F.col("gross_debt_usd"),
        F.col("w.government_revenue_pct_gdp_imf"),
        F.col("w.government_expenditure_pct_gdp_imf"),
        F.col("w.net_lending_borrowing_pct_gdp_imf"),
        F.col("w.current_account_balance_pct_gdp_imf"),
        F.col("loaded_at"),
    )
)

# Anchor check: Nigeria's latest GDP should be hundreds of billions of USD.
# This pins the NGDPD scale assumption empirically instead of trusting comments.
nga = (
    fact_df.where((F.col("country_iso3") == "NGA") & F.col("gdp_current_usd").isNotNull())
    .orderBy(F.desc("year"))
    .select("year", "gdp_current_usd")
    .first()
)
gdp_usd = nga["gdp_current_usd"] if nga else None
assert gdp_usd is not None and 1e11 < gdp_usd < 1e12, (
    f"NGDPD scale assumption looks wrong: NGA GDP came out as {gdp_usd}. "
    "Expected 1e11-1e12 USD (hundreds of billions)."
)
print(f"Scale anchor OK: NGA {nga['year']} GDP = {gdp_usd/1e9:,.0f} B USD")

print(f"Fact table built: {fact_df.count()} rows")
fact_df.orderBy("country_iso3", "year").show(5, truncate=False)


In [ ]:
# Cell 5 - Write silver.fact_macro_annual
spark.sql("DROP TABLE IF EXISTS silver.fact_macro_annual")

(fact_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_macro_annual"))

print("Write complete: silver.fact_macro_annual")
print(f"  Rows: {spark.table('silver.fact_macro_annual').count():,}")


In [ ]:
# Cell 6 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(gdp_current_usd) AS rows_with_gdp,
        COUNT(population_millions) AS rows_with_population,
        COUNT(inflation_cpi_pct) AS rows_with_inflation,
        COUNT(gross_debt_pct_gdp_imf) AS rows_with_debt
    FROM silver.fact_macro_annual
""").show()

print("Coverage by bloc:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS min_year,
        MAX(year) AS max_year,
        ROUND(AVG(gdp_current_usd_billions), 1) AS avg_gdp_usd_b,
        ROUND(AVG(real_gdp_growth_pct_imf), 2) AS avg_gdp_growth_pct
    FROM silver.fact_macro_annual
    GROUP BY analytical_bloc_code
    ORDER BY analytical_bloc_code
""").show(truncate=False)

print("Latest available year per country:")
spark.sql("""
    SELECT country_iso3, analytical_bloc_code,
           MAX(year) AS latest_year,
           ROUND(MAX(gdp_current_usd_billions), 2) AS latest_gdp_usd_b
    FROM silver.fact_macro_annual
    WHERE gdp_current_usd_billions IS NOT NULL
    GROUP BY country_iso3, analytical_bloc_code
    ORDER BY analytical_bloc_code, country_iso3
""").show(25, truncate=False)
